In [4]:
import pandas as pd
import numpy as np

# Loading the dataset
file_path = '/content/Flight_Operations_Dataset.csv'
df = pd.read_csv(file_path)

display(df.head())

,Flight_ID,Flight_Date,Airline,Origin,Destination,Aircraft,Travel_Class,Passengers,Seat_Capacity,Average_Ticket_Price,Delay_Minutes,Flight_Status,Weather,Booking_Channel,Avg_Baggage_Kg,Meal_Preference,Passenger_Satisfaction
0,FL0001,2026-03-23,SpiceJet,Mumbai,Bengaluru,Airbus A319,Economy,210,180,6894,50.0,Delayed,Storm,Online Travel Portal,23,No Meal,3
1,FL0002,2026-06-07,Akasa Air,Delhi,Mumbai,Boeing 737,Economy,70,210,4468,0.0,On Time,Fog,Travel Agency,16,Vegan,5
2,FL0003,2026-05-12,Air India,Kochi,Delhi,Boeing 737,Economy,67,220,4713,5.0,On Time,Rain,Travel Agency,19,No Meal,2
3,FL0004,2026-05-30,Akasa Air,Srinagar,Delhi,Airbus A319,Economy,153,220,3791,35.0,Delayed,Storm,Travel Agency,18,No Meal,2
4,FL0005,2026-02-27,Air India,Pune,Delhi,Airbus A319,Premium Economy,178,180,6505,20.0,Delayed,Cloudy,Airline Website,25,No Meal,5


In [5]:
print("--- Dataset Info ---")
df.info()

print("\n--- Summary Statistics ---")
display(df.describe())

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Flight_ID               180 non-null    object 
 1   Flight_Date             180 non-null    object 
 2   Airline                 180 non-null    object 
 3   Origin                  180 non-null    object 
 4   Destination             180 non-null    object 
 5   Aircraft                180 non-null    object 
 6   Travel_Class            180 non-null    object 
 7   Passengers              180 non-null    int64  
 8   Seat_Capacity           180 non-null    int64  
 9   Average_Ticket_Price    180 non-null    int64  
 10  Delay_Minutes           173 non-null    float64
 11  Flight_Status           180 non-null    object 
 12  Weather                 180 non-null    object 
 13  Booking_Channel         180 non-null    object 
 14  Avg_Baggage_Kg       

,Passengers,Seat_Capacity,Average_Ticket_Price,Delay_Minutes,Avg_Baggage_Kg,Passenger_Satisfaction
count,180.000000,180.000000,180.000000,173.000000,180.000000,180.000000
mean,122.772222,197.238889,6111.677778,19.335260,15.150000,3.494444
std,45.635651,15.408176,3615.220040,29.029712,5.705506,1.169908
min,45.000000,180.000000,1949.000000,0.000000,5.000000,2.000000
25%,78.750000,186.000000,3815.500000,0.000000,10.000000,2.000000
50%,125.000000,189.000000,5380.000000,5.000000,15.500000,3.000000
75%,162.250000,210.000000,6790.000000,20.000000,20.000000,5.000000
max,210.000000,220.000000,27832.000000,110.000000,25.000000,5.000000


In [6]:
df['Delay_Minutes'] = df['Delay_Minutes'].fillna(0)

df['Flight_Date'] = pd.to_datetime(df['Flight_Date'])

df['Load_Factor'] = (df['Passengers'] / df['Seat_Capacity']) * 100

df['Total_Revenue'] = df['Passengers'] * df['Average_Ticket_Price']

def categorize_delay(minutes):
    if minutes <= 15:
        return 'On Time'
    elif minutes <= 60:
        return 'Moderate Delay'
    else:
        return 'Severe Delay'

df['Delay_Category'] = df['Delay_Minutes'].apply(categorize_delay)

print("Data cleaning and transformations complete. Preview of new columns:")
display(df[['Flight_ID', 'Load_Factor', 'Total_Revenue', 'Delay_Category']].head())

Data cleaning and transformations complete. Preview of new columns:


,Flight_ID,Load_Factor,Total_Revenue,Delay_Category
0,FL0001,116.666667,1447740,Moderate Delay
1,FL0002,33.333333,312760,On Time
2,FL0003,30.454545,315771,On Time
3,FL0004,69.545455,580023,Moderate Delay
4,FL0005,98.888889,1157890,Moderate Delay


In [7]:
severe_delays = df[df['Delay_Minutes'] > 60]
print(f"Total flights with severe delays: {len(severe_delays)}")
display(severe_delays[['Airline', 'Origin', 'Destination', 'Delay_Minutes', 'Weather']].head())

business_high_load = df[(df['Travel_Class'] == 'Business') & (df['Load_Factor'] >= 85.0)]
print(f"\nHigh-performing Business Class flights: {len(business_high_load)}")

Total flights with severe delays: 18


,Airline,Origin,Destination,Delay_Minutes,Weather
8,Akasa Air,Delhi,Chennai,75.0,Cloudy
21,Vistara,Kochi,Delhi,75.0,Clear
26,SpiceJet,Bengaluru,Hyderabad,110.0,Clear
29,SpiceJet,Delhi,Srinagar,75.0,Storm
50,Air India,Delhi,Mumbai,110.0,Rain



High-performing Business Class flights: 3


In [9]:
weather_impact = df.groupby('Weather').agg(
    Average_Delay=('Delay_Minutes', 'mean'),
    Total_Flights=('Flight_ID', 'count')
).reset_index().round(2)

print("--- Impact of Weather on Flight Delays ---")
display(weather_impact.sort_values(by='Average_Delay', ascending=False))


airline_performance = df.groupby('Airline').agg(
    Total_Revenue=('Total_Revenue', 'sum'),
    Average_Satisfaction=('Passenger_Satisfaction', 'mean'),
    Average_Load_Factor=('Load_Factor', 'mean')
).reset_index().round(2)

print("\n--- Airline Performance Metrics ---")
display(airline_performance.sort_values(by='Total_Revenue', ascending=False))

--- Impact of Weather on Flight Delays ---


,Weather,Average_Delay,Total_Flights
3,Rain,21.19,43
4,Storm,19.43,30
1,Cloudy,18.45,38
0,Clear,18.03,34
2,Fog,15.34,35



--- Airline Performance Metrics ---


,Airline,Total_Revenue,Average_Satisfaction,Average_Load_Factor
5,Vistara,28689467,3.38,61.87
3,IndiGo,25709131,3.34,66.78
4,SpiceJet,24195935,3.53,63.60
1,Air India Express,23944164,3.67,64.79
2,Akasa Air,18872059,3.42,55.21
0,Air India,16232418,3.75,64.66


In [10]:
top_rated_flights = df.sort_values(by=['Passenger_Satisfaction', 'Delay_Minutes'], ascending=[False, True])

print("--- Top 5 Best Rated Flights ---")
display(top_rated_flights[['Flight_ID', 'Airline', 'Passenger_Satisfaction', 'Delay_Minutes', 'Delay_Category']].head())

--- Top 5 Best Rated Flights ---


,Flight_ID,Airline,Passenger_Satisfaction,Delay_Minutes,Delay_Category
1,FL0002,Akasa Air,5,0.0,On Time
7,FL0008,Air India,5,0.0,On Time
12,FL0013,SpiceJet,5,0.0,On Time
13,FL0014,Air India,5,0.0,On Time
14,FL0015,Akasa Air,5,0.0,On Time


### <b> Data-Driven Observations & Conclusions</b>

Based on the data analysis, grouping, and transformations performed, here are the key findings from the Flight Operations dataset:

1. **Revenue vs. Satisfaction Disconnect:** **Vistara** is the top revenue generator (approx. 28.6 million), but it struggles with customer sentiment, holding the second-lowest average satisfaction score (3.38). In stark contrast, **Air India** generated the lowest total revenue but boasts the highest passenger satisfaction (3.75).
2. **Weather Impacts are Counterintuitive:** As expected, **Rain** causes the highest average delays (21.19 minutes). However, surprisingly, **Fog** resulted in the lowest average delay (15.34 minutes). This suggests airlines may be proactively canceling flights during fog rather than delaying them, or airports have highly efficient instrument landing systems.
3. **Punctuality Drives Perfection:** There is a direct link between zero delays and maximum customer happiness. The top 5 highest-rated flights (scoring a perfect 5.0) all had exactly 0.0 minutes of delay, proving that on-time performance is the biggest driver of passenger satisfaction.
4. **Capacity Utilization (Load Factor):** **IndiGo** is the most efficient at filling its planes, leading the pack with an average load factor of 66.78%. On the other hand, **Akasa Air** struggles with empty seats, showing the lowest average load factor at just 55.21%.
5. **Overbooking Practices Detected:** The data transformations revealed an interesting anomaly in `FL0001`, which has a `Load_Factor` of 116.67%. This indicates the flight was overbooked (selling more tickets than actual seat capacity), which is a common but risky airline industry strategy to maximize revenue.
6. **Severe Delays Aren't Always Weather-Related:** Out of 180 flights, 18 suffered "Severe Delays" of over 60 minutes. Interestingly, the data shows that the absolute longest delays in the dataset (110 minutes on SpiceJet and Air India) occurred during **Clear** weather, pointing to mechanical, operational, or air traffic control issues rather than weather disruptions.
7. **Premium Cabins rarely fly full:** When filtering the dataset, we found only **3 Business Class flights** that achieved a load factor of 85% or higher. This shows that premium cabins rarely operate at near-full capacity compared to standard economy classes.